In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
PRICE_PROMO_DATA = "/Users/solveigroendalliniger/Desktop/Price_Promo.xlsx"

price_promo_workbook = pd.ExcelFile(PRICE_PROMO_DATA)
price_promo_sheet_name = price_promo_workbook.sheet_names[2]

df = pd.read_excel(price_promo_workbook, sheet_name=price_promo_sheet_name)
df.columns = df.columns.astype(str).str.strip()

for col in ["WeekNum", "RRP_Brand_1", "Discount_Brand_1", "Promo_Brand_1",
            "RRP_Brand_2", "Discount_Brand_2", "Promo_Brand_2"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["WeekNum"]).copy()
df["WeekNum"] = df["WeekNum"].astype(int)
df["Promo_Brand_1"] = df["Promo_Brand_1"].fillna(0).astype(int)
df["Promo_Brand_2"] = df["Promo_Brand_2"].fillna(0).astype(int)

df["ActualPrice_Brand_1"] = np.where(df["Promo_Brand_1"] == 1, df["Discount_Brand_1"], df["RRP_Brand_1"])
df["ActualPrice_Brand_2"] = np.where(df["Promo_Brand_2"] == 1, df["Discount_Brand_2"], df["RRP_Brand_2"])

df = df.sort_values("WeekNum").reset_index(drop=True)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

brand_info = [
    (1, "Glyngøre",  "ActualPrice_Brand_1", "RRP_Brand_1", "Promo_Brand_1", "#1c485f", "#a8c8e0"),
    (2, "Navito",    "ActualPrice_Brand_2", "RRP_Brand_2", "Promo_Brand_2", "#8b1a1a", "#f0b8b8"),
]

for ax, (brand_id, brand_name, price_col, rrp_col, promo_col, color, promo_color) in zip(axes, brand_info):
    promo_weeks = df.loc[df[promo_col] == 1, "WeekNum"]
    for w in promo_weeks:
        ax.axvspan(w - 0.5, w + 0.5, color=promo_color, alpha=0.4, linewidth=0)

    ax.plot(df["WeekNum"], df[rrp_col], color="grey", linewidth=1,
            linestyle="--", label="RRP", zorder=2)
    ax.plot(df["WeekNum"], df[price_col], color=color, linewidth=1.5,
            label="Faktisk pris", zorder=3)

    ax.set_title(f"Brand {brand_id}: {brand_name}", fontsize=11)
    ax.set_ylabel("Pris (DKK)")
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

    ax.plot([], [], color=promo_color, linewidth=8, alpha=0.4, label="Kampagneuge")
    ax.legend(loc="upper right", fontsize=9)

axes[-1].set_xlabel("Uge")
fig.suptitle("Prisudvikling over tid", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("nfxp_empirical_price_development.pdf", bbox_inches="tight")
plt.show()